## 1. New Experiment

Keeping the same trained model on MNAR (reproducibility experiment). MNAR: Missing Not At Random, white pixels are two times more likely to be missing. 

### 1.1. Compute statistics on the missing data given by the authors

In [2]:
import numpy as np

hmnist_miss = np.load('../external/datasets/hmnist/x_miss_nan.npy')
hmnist_miss = hmnist_miss.reshape(784, 70_000, 10).transpose(1,2,0)
mask = np.isnan(hmnist_miss)
hmnist_miss[mask] = 0.0

### 1.2. Authors' HMNIST MNAR

In [ ]:
import numpy as np
from utils import create_imputation_plot, compute_mask_statistics, compute_mask_statistics_v2

import sys
sys.path.append('..')
from imputegap.recovery.manager import TimeSeries
from imputegap.algorithms.gp_vae import gp_vae

# initialize the time series object
ts = TimeSeries()

seq_length = 10
nbr_features = 784

# model folder
# model_folder = 'imputegap_assets/models/20251205_152733'
# model_folder = 'external/models/251113_reproduce_hmnist'
model_folder = '/home/kaeslin/Downloads/hmnist_trained'
validation_idx = 600_000
nr_samples_test = 122
sample_idx = 61

hmnist_miss_val = np.load("../external/datasets/hmnist/x_miss_nan.npy")[:,validation_idx:validation_idx+seq_length*nr_samples_test]
hmnist_full_val_gt = np.load("../external/datasets/hmnist/x_full.npy")[:,validation_idx:validation_idx+seq_length*nr_samples_test]

# load the dataset
ts.import_matrix(hmnist_miss_val)

ground_truth = hmnist_full_val_gt.T.reshape(-1,seq_length,nbr_features)

# auroc computation doesn't make sense, since train a logistic regression on 122 samples does 
# not return a good result (switched to a single sample also here)
# y_val = np.load('../external/hmnist/hmnist_mnar.npz')['y_test'][:nr_samples_test]

try:
  ts_m_imputed, extras = gp_vae(ts.data[:, sample_idx*seq_length:sample_idx*seq_length+seq_length], 
                        "../imputegap/wrapper/AlgoPython/GPVAE/config_gpvae_hmnist.yaml", 
                        model_folder,
                        ground_truth=ground_truth[sample_idx:sample_idx+1],
                        return_no_gt_imputation=True,
                        # y_val=y_val,
                        verbose=False)

  print(extras)
  print(compute_mask_statistics_v2(np.isnan(hmnist_miss_val.T.reshape(-1, 10, 784))))
  print(compute_mask_statistics_v2(np.isnan(hmnist_miss_val.T.reshape(-1, 10, 784)), ground_truth, target_value=1))

  
  create_imputation_plot((28,28,1),
                       10, 
                       ts.data[:, sample_idx*seq_length:sample_idx*seq_length+seq_length].T.reshape(-1, seq_length, nbr_features),
                       extras['recov_no_gt'].T.reshape(-1, seq_length, nbr_features),
                       ts_m_imputed.T.reshape(-1, seq_length, nbr_features),
                       ground_truth[sample_idx:sample_idx+1],
                       0)  
finally:
  ts=None
  hmnist_miss_val = None
  hmnist_full_val_gt = None

Found config file in the checkpoints folder, it is going to be used instead of the one provided as config_yaml_path folder config_gpvae_hmnist.yaml
Checkpoint successfully restored.


100%|██████████| 1/1 [00:00<00:00, 12.07it/s]


{'recov_no_gt': array([[4.8957106e-12, 6.7789635e-11, 3.1230569e-09, ..., 1.5143542e-12,
        1.2659565e-12, 3.9552263e-12],
       [1.8286503e-12, 4.3324171e-11, 7.4495143e-10, ..., 1.5031083e-12,
        1.2838927e-12, 2.0902412e-12],
       [4.8445102e-12, 5.3436307e-11, 8.1076840e-10, ..., 2.5853605e-12,
        1.2181903e-12, 4.3555398e-12],
       ...,
       [1.0662721e-12, 5.9605541e-12, 7.6089712e-10, ..., 5.5934955e-13,
        2.7799513e-13, 3.8264707e-13],
       [1.1144827e-12, 2.1965004e-11, 3.0342059e-10, ..., 4.9554705e-12,
        2.0485850e-12, 3.5015775e-12],
       [2.3310587e-12, 5.5135816e-11, 6.1010264e-10, ..., 2.7804085e-12,
        7.0795447e-13, 1.2088392e-12]], dtype=float32), 'evaluation_metrics': {'nll': 0.45745146901983963, 'mse': 0.14035087719298245}}
((0.44792572766811645, 0.022531871365504545), (0.4479257276681164, 0.015794818474347896), 0.44792572766811645)
((0.7985515422503437, 0.04455509156589401), (0.7985603322506796, 0.014754206950447825), 0.79

### 1.3. Custom Contaminations

To contaminate effectively I need more data, seq_length of 10 is too small, already with the offset the start of the sequence is not contaminated. Solution, contaminate a larger set of data, and then select to visualize the imputation, in this way however, the nll and mse are not probably representative of the contamination. I think I could then save the contamination of the specific sample, and evaluate it individually.

Scattered

In [6]:
import numpy as np
from utils import create_imputation_plot, compute_mask_statistics, compute_mask_statistics_v2

import sys
sys.path.append('..')
from imputegap.recovery.manager import TimeSeries
from imputegap.algorithms.gp_vae import gp_vae

# initialize the time series object
ts = TimeSeries()

seq_length = 10
nbr_features = 784

# model folder
# model_folder = 'imputegap_assets/models/20251205_152733'
# model_folder = 'external/models/251113_reproduce_hmnist'
model_folder = '/home/kaeslin/Downloads/hmnist_trained'


validation_idx = 600_000
nr_samples_test = 122
sample_idx = 61

rate_dataset = 1.0
rate_series = 0.3
offset=0.1

hmnist_full_val_gt = np.load("../external/datasets/hmnist/x_full.npy")[:,validation_idx:validation_idx+seq_length*nr_samples_test]

# load the dataset
ts.import_matrix(hmnist_full_val_gt)

print(ts.data.shape)
# Add missingness to the data, ts has shape (T, V)
ts_m = ts.Contamination.scattered(ts.data,rate_dataset=rate_dataset,
                                  rate_series=rate_series, 
                                  offset=offset)

ground_truth = hmnist_full_val_gt.T.reshape(-1,seq_length,nbr_features)

try:
  ts_m_imputed, extras = gp_vae(ts_m[:, sample_idx*seq_length:sample_idx*seq_length+seq_length], 
                      "../imputegap/wrapper/AlgoPython/GPVAE/config_gpvae_hmnist.yaml", 
                      model_folder,
                      ground_truth=ground_truth[sample_idx:sample_idx+1],
                      return_no_gt_imputation=True,
                      verbose=False)

  print(extras)
  print(compute_mask_statistics_v2(np.isnan(ts_m[:, sample_idx*seq_length:sample_idx*seq_length+seq_length].T.reshape(-1, 10, 784))))
  print(compute_mask_statistics_v2(np.isnan(ts_m[:, sample_idx*seq_length:sample_idx*seq_length+seq_length].T.reshape(-1, 10, 784)), ground_truth, target_value=1))

  create_imputation_plot((28,28,1),
                        10, 
                        ts_m[:, sample_idx*seq_length:sample_idx*seq_length+seq_length].T.reshape(-1, seq_length, nbr_features),
                        extras['recov_no_gt'].T.reshape(-1, seq_length, nbr_features),
                        ts_m_imputed.T.reshape(-1, seq_length, nbr_features),
                        ground_truth[sample_idx:sample_idx+1],
                        0)
finally:
  ts=None
  hmnist_miss_val = None
  hmnist_full_val_gt = None

(784, 1220)

(CONT) missigness pattern: SCATTER
	percentage of contaminated series: 100.0%
	rate of missing data per series: 30.0%
	security offset: [0-122]
	index impacted : 122 -> 488
Found config file in the checkpoints folder, it is going to be used instead of the one provided as config_yaml_path folder config_gpvae_hmnist.yaml
Checkpoint successfully restored.


100%|██████████| 1/1 [00:00<00:00,  8.80it/s]


{'recov_no_gt': array([[1.35938602e-12, 7.62700667e-13, 1.22020007e-11, ...,
        2.34200419e-12, 1.78433335e-14, 1.65469432e-15],
       [3.87767997e-13, 2.50601796e-12, 1.10032339e-11, ...,
        7.94086693e-13, 5.19106217e-15, 3.27822185e-16],
       [5.11179586e-13, 5.31986944e-13, 8.48159910e-12, ...,
        9.14514883e-13, 7.58126844e-15, 1.28167083e-16],
       ...,
       [3.14537242e-13, 1.06370194e-13, 5.02926607e-12, ...,
        5.83274531e-13, 4.17646916e-15, 6.47478271e-17],
       [2.23524947e-13, 3.54147945e-13, 1.96455525e-12, ...,
        2.10472351e-12, 2.11251965e-14, 6.38696406e-16],
       [2.42501711e-13, 7.57685310e-13, 4.79737604e-12, ...,
        1.15324493e-12, 1.01845649e-14, 3.60193349e-16]], dtype=float32), 'evaluation_metrics': {'nll': 0.15904548617305073, 'mse': 0.06123508043591074}}
((0.4915816326530612, 0.0010204081632653197), (0.4915816326530612, 0.0), 0.4915816326530612)
((0.4787988585973738, 0.048584464189767314), (0.4787624715616694, 0.026341

MCAR

In [7]:
import numpy as np
from utils import create_imputation_plot, compute_mask_statistics_v2

import sys
sys.path.append('..')
from imputegap.recovery.manager import TimeSeries
from imputegap.algorithms.gp_vae import gp_vae

# initialize the time series object
ts = TimeSeries()

seq_length = 10
nbr_features = 784

# model folder
# model_folder = 'imputegap_assets/models/20251205_152733'
# model_folder = 'external/models/251113_reproduce_hmnist'
model_folder = '/home/kaeslin/Downloads/hmnist_trained'


validation_idx = 600_000
sample_idx = 61
nr_samples_test = 122

# contamination parameters
rate_dataset = 1.0
rate_series = 0.8
block_size = 3

hmnist_full_val_gt = np.load("../external/datasets/hmnist/x_full.npy")[:,validation_idx:validation_idx+seq_length*nr_samples_test]

# load the dataset
ts.import_matrix(hmnist_full_val_gt)

print(ts.data.shape)
# Add missingness to the data, ts has shape (T, V)
ts_m = ts.Contamination.mcar(ts.data,
                             rate_dataset=rate_dataset, 
                             rate_series=rate_series, block_size=block_size)

ground_truth = hmnist_full_val_gt.T.reshape(-1,seq_length,nbr_features)

try:
  ts_m_imputed, extras = gp_vae(ts_m[:, sample_idx*seq_length:sample_idx*seq_length+seq_length], 
                      "../imputegap/wrapper/AlgoPython/GPVAE/config_gpvae_hmnist.yaml", 
                      model_folder,
                      ground_truth=ground_truth[sample_idx:sample_idx+1],
                      return_no_gt_imputation=True,
                      verbose=False)

  print(extras)
  print(compute_mask_statistics_v2(np.isnan(ts_m[:, sample_idx*seq_length:sample_idx*seq_length+seq_length].T.reshape(-1, 10, 784))))
  print(compute_mask_statistics_v2(np.isnan(ts_m[:, sample_idx*seq_length:sample_idx*seq_length+seq_length].T.reshape(-1, 10, 784)), ground_truth, target_value=1))

  create_imputation_plot((28,28,1),
                        10, 
                        ts_m[:, sample_idx*seq_length:sample_idx*seq_length+seq_length].T.reshape(-1, seq_length, nbr_features),
                        extras['recov_no_gt'].T.reshape(-1, seq_length, nbr_features),
                        ts_m_imputed.T.reshape(-1, seq_length, nbr_features),
                        ground_truth[sample_idx:sample_idx+1],
                        0)
finally:
  ts=None
  hmnist_miss_val = None
  hmnist_full_val_gt = None

(784, 1220)

(CONT) missigness pattern: MCAR
	selected series: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 

100%|██████████| 1/1 [00:00<00:00, 11.92it/s]


{'recov_no_gt': array([[4.08049415e-14, 1.24023933e-13, 2.55900665e-11, ...,
        1.57555338e-12, 7.08047024e-11, 7.39698164e-11],
       [1.07311815e-14, 6.16562397e-14, 1.33222704e-11, ...,
        9.04685354e-12, 6.51564774e-11, 1.85849738e-11],
       [6.82428086e-14, 1.84445815e-13, 3.16139823e-11, ...,
        5.05664087e-12, 8.46844539e-11, 2.21769617e-11],
       ...,
       [2.95919024e-14, 2.58552555e-14, 9.32698363e-12, ...,
        2.85587553e-13, 1.14253936e-11, 7.75254340e-12],
       [3.74362203e-14, 5.64663909e-14, 1.60287061e-11, ...,
        4.56108502e-12, 8.61687804e-11, 2.87772861e-11],
       [5.52244543e-14, 4.68617759e-14, 8.89651026e-12, ...,
        1.62348791e-12, 3.11982662e-11, 1.57947180e-11]], dtype=float32), 'evaluation_metrics': {'nll': 0.6047294941400672, 'mse': 0.13867466438160525}}
((0.8931122448979592, 0.007981881554674487), (0.8931122448979592, 0.0), 0.8931122448979592)
((0.8831575011187768, 0.035708364036102266), (0.8831874080577712, 0.01344508